# Retrieval + RAG (Offline-First)

This notebook builds retrieval pipelines without any network calls (except the model).
We compare sparse (BM25) and deterministic dense-like retrieval with `DeterministicFakeEmbedding`.

**Goals**
- Build offline retrievers.
- Assemble a minimal RAG chain in LCEL.
- Inspect retrieved context to catch common mistakes.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openrouter import ChatOpenRouter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
model = ChatOpenRouter(model=OPENROUTER_MODEL, temperature=0.2)
parser = StrOutputParser()

## Local corpus
We will use a small local corpus to keep the pipeline deterministic.

In [2]:
docs = [
    Document(page_content="LangChain is a framework for building LLM applications with composable primitives.", metadata={"source": "doc1"}),
    Document(page_content="RAG combines retrieval with generation by injecting relevant context into prompts.", metadata={"source": "doc2"}),
    Document(page_content="LCEL is the LangChain Expression Language for composing runnables.", metadata={"source": "doc3"}),
    Document(page_content="Retrievers return Documents for a query; vector stores are just one retriever source.", metadata={"source": "doc4"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10)
splits = splitter.split_documents(docs)
len(splits)

7

## Sparse retrieval (BM25)
BM25 is simple, offline, and sometimes outperforms naive embeddings on small corpora.

In [3]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(splits)
bm25.invoke("What is LCEL?")

[Document(metadata={'source': 'doc3'}, page_content='LCEL is the LangChain Expression Language for composing runnables.'),
 Document(metadata={'source': 'doc1'}, page_content='LangChain is a framework for building LLM applications with composable'),
 Document(metadata={'source': 'doc4'}, page_content='retriever source.'),
 Document(metadata={'source': 'doc4'}, page_content='Retrievers return Documents for a query; vector stores are just one retriever')]

## Deterministic dense-like retrieval (offline)
Use `DeterministicFakeEmbedding` with `InMemoryVectorStore` to keep a vector-based flow without any API calls.

In [4]:
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = DeterministicFakeEmbedding(size=64)
vectorstore = InMemoryVectorStore(embeddings)
_ = vectorstore.add_documents(splits)

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
dense_retriever.invoke("What does a retriever do?")

[Document(id='49f7f79b-41b0-411c-baee-c72c46c76d22', metadata={'source': 'doc1'}, page_content='primitives.'),
 Document(id='15feb76f-fd3a-4562-8545-609c8e16c8db', metadata={'source': 'doc4'}, page_content='Retrievers return Documents for a query; vector stores are just one retriever')]

## Minimal RAG chain (LCEL)
We will plug the BM25 retriever into the chain to keep the pipeline fully offline except for the model.

In [14]:
rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below.\n\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {"context": dense_retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | parser
)

rag_chain.invoke("What does a retriever do?")

'Based on the provided context, a retriever returns Documents for a query.'

## Retrieval sanity checks
Always inspect what your retriever actually returns.

In [15]:
def show_sources(docs):
    return [(d.metadata.get("source"), d.page_content) for d in docs]

show_sources(bm25.invoke("retriever vs vector store"))

[('doc4',
  'Retrievers return Documents for a query; vector stores are just one retriever'),
 ('doc4', 'retriever source.'),
 ('doc3',
  'LCEL is the LangChain Expression Language for composing runnables.'),
 ('doc2', 'into prompts.')]

## Common misuses to watch for
- Stuffing too much context into prompts (no chunking).
- Using a retriever but never inspecting its outputs.
- Confusing vector stores with retrievers (retrievers are the runtime interface).
- Returning answer text without showing provenance in dev.